# 01. EDA — 생활 쓰레기 12클래스

데이터: Kaggle `mostafaabla/garbage-classification`

**이 노트북에서 확인할 것**
1. 경로 및 클래스 구조
2. 클래스 분포 (불균형 정도)
3. 손상 파일 · 이미지 모드 · 해상도
4. 클래스별 샘플 육안 확인
5. **유리 3종(brown/green/white) 색상 분리도 검증** ← 핵심 가설
6. 다음 단계로 넘길 index.csv 저장

## 1. 임포트 및 경로 확인

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

# 노트북이 17-DeepLearning 루트에 있다는 전제
DATA = Path("data/garbage_classification")
OUT  = Path("outputs/garbage")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "metrics").mkdir(parents=True, exist_ok=True)
(OUT / "models").mkdir(parents=True, exist_ok=True)

print("경로 존재:", DATA.exists())
for p in sorted(DATA.iterdir())[:15]:
    print("   ", p.name, "/" if p.is_dir() else "")

> **확인**: 위 출력이 `battery`, `biological`, `brown-glass` … 처럼 **클래스 폴더**로 나와야 합니다.
> 만약 `garbage_classification` 하나만 나오면 캐글 zip이 한 겹 더 감싼 구조이므로 아래를 실행하세요.

```python
DATA = DATA / "garbage_classification"
```

## 2. 파일 인덱스 & 클래스 분포

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = [(d.name, str(f))
        for d in sorted(DATA.iterdir()) if d.is_dir()
        for f in sorted(d.iterdir()) if f.suffix.lower() in EXT]

df = pd.DataFrame(rows, columns=["label", "path"])
classes = sorted(df["label"].unique())

cnt = df["label"].value_counts()
imbalance = cnt.max() / cnt.min()

print(f"총 {len(df):,}장 / {len(classes)}클래스")
print(f"최다 {cnt.idxmax()} {cnt.max():,}장  |  최소 {cnt.idxmin()} {cnt.min():,}장")
print(f"불균형 비율: {imbalance:.1f}배")
print()
print(cnt.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#c0392b" if v == cnt.max() or v == cnt.min() else "#4a7ba7" for v in cnt.values]
ax.barh(cnt.index, cnt.values, color=colors)
ax.invert_yaxis()
ax.set_xlabel("images")
ax.set_title(f"Class distribution  (imbalance {imbalance:.1f}x)")
for i, v in enumerate(cnt.values):
    ax.text(v + 40, i, f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUT / "figures" / "01_class_dist.png", dpi=150)
plt.show()

**해석 포인트 (발표용)**

불균형이 크면 accuracy는 다수 클래스에 끌려갑니다. 최다 클래스만 전부 맞히고 나머지를 다 틀려도
겉보기 정확도가 나오므로, 평가 지표를 **macro-F1 + 클래스별 recall**로 잡아야 하는 근거가 여기서 나옵니다.
대응책(WeightedRandomSampler / class_weight) 비교는 `03`, `04` 노트북에서 진행합니다.

## 3. 손상 파일 · 이미지 모드 · 해상도

In [ ]:
sizes, modes, broken = [], Counter(), []

for p in df["path"]:
    try:
        with Image.open(p) as im:
            im.verify()              # 손상 여부 검사
        with Image.open(p) as im:    # verify 후에는 재오픈 필요
            sizes.append(im.size)
            modes[im.mode] += 1
    except Exception as e:
        broken.append((p, repr(e)))

print(f"손상 파일: {len(broken)}개")
for b in broken[:10]:
    print("   ", b)

print("\n이미지 모드 분포:", dict(modes))

s = pd.DataFrame(sizes, columns=["w", "h"])
print("\n해상도 요약")
print(s.describe().loc[["min", "25%", "50%", "75%", "max"]].astype(int).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(s["w"], bins=40, color="#4a7ba7")
axes[0].set_title("Width distribution"); axes[0].set_xlabel("px")

axes[1].scatter(s["w"], s["h"], s=3, alpha=0.15, color="#4a7ba7")
axes[1].set_title("Width vs Height"); axes[1].set_xlabel("width"); axes[1].set_ylabel("height")

plt.tight_layout()
plt.savefig(OUT / "figures" / "02_resolution.png", dpi=150)
plt.show()

**해석 포인트**

- 모드에 `L`(흑백)·`RGBA`·`P`가 섞여 있으면 텐서 채널 수가 달라져 학습 중 터집니다.
  → 전처리에서 `.convert("RGB")`를 **반드시** 넣는 근거입니다.
- 해상도가 제각각이면 `Resize`가 필수입니다. 대부분 작은 이미지면 224로 키울 때
  업스케일 손실이 생기므로 이 분포를 근거로 입력 크기를 정합니다.

## 4. 클래스별 샘플 육안 확인

In [ ]:
N_SHOW = 5
fig, axes = plt.subplots(len(classes), N_SHOW, figsize=(2.0 * N_SHOW, 2.0 * len(classes)))

for r, c in enumerate(classes):
    paths = random.sample(list(df.loc[df.label == c, "path"]), N_SHOW)
    for j, p in enumerate(paths):
        ax = axes[r, j]
        ax.imshow(Image.open(p).convert("RGB"))
        ax.axis("off")
        if j == 0:
            ax.set_title(c, loc="left", fontsize=10, color="#c0392b")

plt.tight_layout()
plt.savefig(OUT / "figures" / "03_samples.png", dpi=120)
plt.show()

## 5. 유리 3종 색상 분리도 검증 (OpenCV)

**가설**: `brown-glass` / `green-glass` / `white-glass`는 **형태가 아니라 색으로만** 구분된다.
따라서 색 정보가 실제로 세 클래스를 가르는지 확인해야 하고, 겹친다면 혼동행렬에서
이 3×3 블록이 오차의 주범이 될 것이다.

강의(`2_이미지_처리_기초`)에서 다룬 `cv2.cvtColor` 색공간 변환을 그대로 사용합니다.
RGB 대신 **HSV**를 쓰는 이유는 색상(Hue)과 밝기(Value)가 분리되어,
조명 차이에 덜 흔들리면서 "무슨 색인가"만 뽑아낼 수 있기 때문입니다.

In [ ]:
GLASS = [c for c in classes if "glass" in c]
print("대상 클래스:", GLASS)

N_SAMPLE = 250     # 클래스당 표본 수
RESIZE   = (96, 96)
SAT_MIN  = 40      # 채도가 너무 낮은 픽셀(회색·흰 배경)은 색상값이 무의미하므로 제외
VAL_MIN  = 30      # 너무 어두운 픽셀도 제외

def hsv_stats(path):
    """이미지 1장 → (hue 히스토그램 36bin, 평균 채도, 평균 명도)"""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.resize(img, RESIZE)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)      # H:0~179, S:0~255, V:0~255
    h, sat, val = hsv[..., 0], hsv[..., 1], hsv[..., 2]

    mask = (sat >= SAT_MIN) & (val >= VAL_MIN)
    if mask.sum() < 50:                              # 유효 픽셀이 거의 없으면 무채색
        hist = np.zeros(36, dtype=np.float32)
    else:
        hist, _ = np.histogram(h[mask], bins=36, range=(0, 180))
        hist = hist.astype(np.float32) / hist.sum()

    return hist, sat.mean(), val.mean(), mask.mean()

records, hists = [], {}

for c in GLASS:
    paths = random.sample(list(df.loc[df.label == c, "path"]), 
                          min(N_SAMPLE, (df.label == c).sum()))
    acc = []
    for p in paths:
        r = hsv_stats(p)
        if r is None:
            continue
        hist, ms, mv, mratio = r
        acc.append(hist)
        records.append({"label": c, "mean_sat": ms, "mean_val": mv, "color_ratio": mratio})
    hists[c] = np.mean(acc, axis=0)
    print(f"{c:14s} 표본 {len(acc)}장")

stat = pd.DataFrame(records)
print()
print(stat.groupby("label")[["mean_sat", "mean_val", "color_ratio"]].mean().round(1).to_string())

In [ ]:
bin_centers = np.arange(36) * 5 + 2.5      # OpenCV Hue 0~179 기준
palette = {"brown-glass": "#8B4513", "green-glass": "#2E8B57", "white-glass": "#7f8c8d"}

fig, ax = plt.subplots(figsize=(10, 4.5))
for c in GLASS:
    ax.plot(bin_centers, hists[c], label=c, lw=2, color=palette.get(c))
    ax.fill_between(bin_centers, hists[c], alpha=0.15, color=palette.get(c))

ax.set_xlabel("Hue (OpenCV scale, 0-179)")
ax.set_ylabel("normalized frequency")
ax.set_title("Average hue distribution of glass classes")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "04_glass_hue.png", dpi=150)
plt.show()

In [ ]:
def overlap(h1, h2):
    """두 정규화 히스토그램의 겹침 계수 (0=완전 분리, 1=완전 동일)"""
    return float(np.minimum(h1, h2).sum())

print("Hue 히스토그램 겹침 계수")
print("-" * 42)
pairs = [(a, b) for i, a in enumerate(GLASS) for b in GLASS[i+1:]]
for a, b in pairs:
    ov = overlap(hists[a], hists[b])
    flag = "  <-- 혼동 위험" if ov > 0.5 else ""
    print(f"{a:14s} vs {b:14s}  {ov:.3f}{flag}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for c in GLASS:
    sub = stat[stat.label == c]
    ax.scatter(sub["mean_sat"], sub["mean_val"], s=18, alpha=0.5,
               label=c, color=palette.get(c))

ax.set_xlabel("mean Saturation")
ax.set_ylabel("mean Value (brightness)")
ax.set_title("Glass classes in Saturation-Value space")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "05_glass_sat_val.png", dpi=150)
plt.show()

**읽는 법**

- **Hue 곡선이 갈라져 있다** → 색으로 구분 가능. 그럼에도 CNN이 틀린다면 색이 아니라
  배경·조명·촬영 각도에 흔들린 것이므로 `ColorJitter` 증강을 약하게 줘야 합니다.
  (색이 정답 신호인데 색을 흔들어 버리면 오히려 성능이 떨어집니다.)
- **곡선이 겹친다 (겹침 계수 > 0.5)** → 색만으로는 부족. 혼동행렬의 유리 3×3 블록이
  오차의 주범이 될 것이라는 예측을 여기서 미리 세워두고, `05_eval_report`에서 답을 맞춰봅니다.
- `white-glass`는 정의상 무채색이라 **평균 채도가 낮고 color_ratio도 낮게** 나오는 것이 정상입니다.
  이 지표 자체가 white를 가르는 특징이 됩니다.

이 그림이 발표자료에서 "우리는 데이터를 들여다봤다"를 증명하는 슬라이드가 됩니다.

## 6. 다음 단계로 넘길 인덱스 저장

In [ ]:
# 손상 파일 제외
broken_paths = {b[0] for b in broken}
clean = df[~df["path"].isin(broken_paths)].reset_index(drop=True)

clean.to_csv(OUT / "metrics" / "index.csv", index=False, encoding="utf-8")

summary = {
    "total_images":  len(clean),
    "n_classes":     len(classes),
    "broken_files":  len(broken),
    "imbalance":     round(float(imbalance), 2),
    "max_class":     f"{cnt.idxmax()} ({cnt.max()})",
    "min_class":     f"{cnt.idxmin()} ({cnt.min()})",
    "modes":         dict(modes),
}
pd.Series(summary).to_csv(OUT / "metrics" / "eda_summary.csv", encoding="utf-8")

print("index.csv 저장:", len(clean), "행")
for k, v in summary.items():
    print(f"  {k:14s} {v}")

---

## 정리 — 다음 노트북으로 넘기는 결론

| 발견 | `02_split` 이후에서의 대응 |
|---|---|
| 클래스 불균형 | stratified split + 평가지표를 macro-F1로 |
| 이미지 모드 혼재 | 전처리에 `.convert("RGB")` 강제 |
| 해상도 제각각 | `Resize` 필수, 입력 크기 결정 |
| 유리 3종 색상 겹침 | `ColorJitter` 강도 조절, 혼동행렬 집중 관찰 |

→ `03P_02_split.ipynb` 로 이동